# Copyright notice
Download the archive yourself from
https://www.swgdrug.org/IR/JCAMP_051524.zip
and run the processing below on the extracted files.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline

# ========== target wavenumber grid ==========
COMMON_X = np.arange(552, 3844, 2.0, dtype=np.float32)

# ========== read a single JCAMP file (verified) ==========
def read_jcamp_correct(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    def get_param(key):
        match = re.search(f"##{key}=([\\d.]+)", content)
        return float(match.group(1)) if match else None
    delta_x = get_param("DELTAX")
    y_factor = get_param("YFACTOR")
    n_points = int(get_param("NPOINTS") or 0)
    data_part = content.split("##XYDATA=(X++(Y..Y))")[1]
    if "##" in data_part:
        data_part = data_part.split("##")[0]
    x_vals = []
    y_vals = []
    for line in data_part.strip().splitlines():
        if not line.strip():
            continue
        nums = re.findall(r"[\d.]+", line)
        if not nums:
            continue
        x_start = float(nums[0])
        for i, y_str in enumerate(nums[1:]):
            x = x_start + i * delta_x
            y = float(y_str) * y_factor
            x_vals.append(x)
            y_vals.append(y)
    if n_points and len(y_vals) > n_points:
        y_vals = y_vals[:n_points]
        x_vals = x_vals[:n_points]
    elif n_points and len(y_vals) < n_points:
        print(f"WARNING: only {len(y_vals)} points extracted from {os.path.basename(filepath)}, fewer than {n_points}")
    return np.array(x_vals, dtype=np.float32), np.array(y_vals, dtype=np.float32)

# ========== extract the compound name ==========
def extract_compound_name(filename):
    base = re.sub(r"^#\d+\s*-\s*", "", filename)
    base = re.sub(r"\s*\(Lot[^)]*\)", "", base)
    base = re.sub(r"\.JDX$", "", base, flags=re.I)
    return base.strip()

# ========== interpolate a single spectrum (truncate + check range) ==========
def interpolate_spectrum(x, y, target_x, file_name):
    """
    Interpolate a spectrum onto the target_x grid.
    If the original x range does not fully cover [target_x.min(), target_x.max()],
    a warning is printed and only the overlapping part is interpolated; the rest becomes NaN.
    """
    x_min, x_max = x.min(), x.max()
    t_min, t_max = target_x.min(), target_x.max()
    
    # check the coverage
    if x_min > t_min or x_max < t_max:
        print(f"⚠️ {file_name} wavenumber range [{x_min:.2f}, {x_max:.2f}] does not fully cover the target range [{t_min:.2f}, {t_max:.2f}]")
    
    # interpolation interval: the overlap of the original and target ranges
    valid_mask = (x >= t_min) & (x <= t_max)
    if not np.any(valid_mask):
        # no overlap at all, return all NaN
        return np.full_like(target_x, np.nan, dtype=np.float32)
    
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    
    # too few valid points to interpolate
    if len(x_valid) < 2:
        # a single point cannot be splined; simply return all NaN here
        return np.full_like(target_x, np.nan, dtype=np.float32)
    
    # spline interpolation (extrapolation is off by default, but the range is truncated above)
    spline = CubicSpline(x_valid, y_valid, bc_type='natural', extrapolate=False)
    # interpolate only the target_x points inside the original range, set the rest to NaN
    result = spline(target_x)
    # set the target_x points outside the original [x_min, x_max] to NaN (extrapolate=False would raise, so this is done manually)
    # with extrapolate=False the spline raises for out-of-range points, so compute first and mask afterwards
    # safer: interpolate only the target_x points within [x_min, x_max] and set the rest to NaN
    in_range = (target_x >= x_min) & (target_x <= x_max)
    result = np.full_like(target_x, np.nan, dtype=np.float32)
    if np.any(in_range):
        # build the interpolator from all valid data (no truncation), but evaluate it only inside the target range
        # to avoid extrapolation problems all valid data is still used, with a final mask
        spline_full = CubicSpline(x_valid, y_valid, bc_type='natural', extrapolate=True)
        result_full = spline_full(target_x)
        result[in_range] = result_full[in_range]
        # out-of-range values stay NaN (as intended by the truncation)
    return result

# ========== build the spectral matrix ==========
def build_spectral_matrix(folder_path, output_csv=None):
    all_data = []  # holds (compound_name, interpolated_y)
    
    for file in os.listdir(folder_path):
        if not file.lower().endswith('.jdx'):
            continue
        filepath = os.path.join(folder_path, file)
        try:
            x, y = read_jcamp_correct(filepath)
        except Exception as e:
            print(f"skipping {file}, parsing failed: {e}")
            continue
        
        # interpolate onto COMMON_X
        y_interp = interpolate_spectrum(x, y, COMMON_X, file)
        
        name = extract_compound_name(file)
        all_data.append((name, y_interp))
    
    if not all_data:
        raise ValueError("no valid spectral data was read")
    
    # build the DataFrame
    df = pd.DataFrame([y for _, y in all_data], index=[name for name, _ in all_data])
    df.columns = [f"{v:.2f}" for v in COMMON_X]
    df.index.name = "Compound"
    
    if output_csv:
        df.to_csv(output_csv)
        print(f"spectral matrix saved to {output_csv}")
    
    return df

# ========== usage example ==========
if __name__ == "__main__":
    folder = "./JCAMP_051524.extracted"   # change this to the real path
    output = "spectral_matrix_interp.csv"
    df = build_spectral_matrix(folder, output)
    print(f"read {df.shape[0]} samples, {df.shape[1]} wavenumber points each")

In [13]:
# then merge with the provided smiles_result.txt file to obtain the final prediction file
import pandas as pd
spec = pd.read_csv("spectral_matrix_interp.csv")
smiles = pd.read_csv("smiles_result.txt", sep="\t", encoding='utf-8-sig')
spec = spec.drop_duplicates(subset='Compound', keep='first')
merged_data = pd.merge(spec, smiles, left_on='Compound', right_on='Compound', how='inner') 

In [16]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.rdBase import DisableLog

# disable the verbose RDKit logging
DisableLog("rdApp.*")

# ----------------------------
# 1. define the functional-group SMARTS patterns
# ----------------------------
SMARTS_PATTERNS = {
    "alkane": "[CX4]",
    "alkene": "[CX3]=[CX3]",
    "alkyne": "[CX2]#C",
    "aromatics": "[$([cX3](:*):*),$([cX2+](:*):*)]",
    "alkyl_halides": "[#6][F,Cl,Br,I]",
    "alcohols": "[#6][OX2H]",
    "esters": "[#6][CX3](=O)[OX2H0][#6]",
    "ketones": "[#6][CX3](=O)[#6]",
    "aldehydes": "[CX3H1](=O)[#6]",
    "carbonyl_oxygen": "[CX3](=O)[OX2]",
    "ether": "[OX2;!$(OC=O)]([#6])[#6]",
    "acyl_halides": "[CX3](=[OX1])[F,Cl,Br,I]",
    "amines": "[NX3;!$(N=*);!$(N#*);!$(N-C(=O));!$(N-C(=S))]",
    "amides": "[NX3][CX3](=[OX1])[#6]",
    "nitriles": "[NX1]#[CX2]",
    "nitro": "[$([NX3](=O)=O),$([NX3+](=O)[O-])][!#8]",
    "isocyanate": "[#7]=[C]=[O]",
    "isothiocyanate": "[#7]=[C]=[S]",
    "ortho": "*-!:aa-!:*",
    "meta": "*-!:aaa-!:*",
    "para": "*-!:aaaa-!:*",
}
# pre-compile the SMARTS patterns (for speed)
_COMPILED_PATTERNS = {}
for name, smarts in SMARTS_PATTERNS.items():
    pat = Chem.MolFromSmarts(smarts)
    _COMPILED_PATTERNS[name] = pat  # a None pattern yields 0 later on

# ----------------------------
# 2. counting function for a single SMILES
# ----------------------------
def count_functional_groups(smiles: str):
    """
    Take one SMILES string and return a dict mapping functional-group name to occurrence count.
    If the molecule is invalid, every count is 0.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        # return an all-zero dict
        return {name: 0 for name in SMARTS_PATTERNS.keys()}
    
    counts = {}
    for name, pat in _COMPILED_PATTERNS.items():
        if pat is None:
            counts[name] = 0
        else:
            counts[name] = len(mol.GetSubstructMatches(pat))
    return counts

# ----------------------------
# 3. apply it to merged_data
# ----------------------------
# assumes merged_data already exists and has a 'smiles' column
# make sure merged_data is a pandas DataFrame

# approach: row-wise apply (serial, no parallelism)
func_counts = merged_data['smiles'].apply(count_functional_groups)

# convert the list of dicts into a DataFrame
func_df = pd.DataFrame(func_counts.tolist(), index=merged_data.index)

# concatenate the functional-group columns onto the original DataFrame
merged_data_with_features = pd.concat([merged_data, func_df], axis=1)

# ----------------------------
# 4. inspect the result
# ----------------------------
print("original shape:", merged_data.shape)
print("new shape:", merged_data_with_features.shape)
print("new column names:", func_df.columns.tolist())

# to save the result:
# merged_data_with_features.to_csv("merged_with_features.csv", index=False)

original shape: (830, 1648)
new shape: (830, 1669)
new column names: ['alkane', 'alkene', 'alkyne', 'aromatics', 'alkyl_halides', 'alcohols', 'esters', 'ketones', 'aldehydes', 'carbonyl_oxygen', 'ether', 'acyl_halides', 'amines', 'amides', 'nitriles', 'nitro', 'isocyanate', 'isothiocyanate', 'ortho', 'meta', 'para']


In [22]:
def count_compounds(smiles):
    if pd.isna(smiles) or smiles == '':
        return 0
    return smiles.count('.') + 1
merged_data_with_features = merged_data_with_features.copy()
merged_data_with_features['component_count'] = merged_data_with_features['smiles'].apply(count_compounds)

In [25]:
merged_data_with_features['source_name'] = list(['swgdrug'] * len(merged_data_with_features))
#2
spec = merged_data_with_features.loc[:,"552.00":"3842.00"].values
np.nan_to_num(spec, nan=0.0)
merged_data_with_features['spectrum'] = spec.tolist()
merged_data_with_features["_eval_name"] = list(['swgdrug'] * len(merged_data_with_features))

In [ ]:
merged_data.to_csv("full_swgdrug_data.csv", index=False, encoding='utf-8-sig')
print("final data saved to full_swgdrug_data.csv")

final data saved to full_swgdrug_data.csv


# the full experiment can now be run separately with the prediction script